# Scrapping Allociné

In [1]:
import numpy as np, pandas as pd, requests, re, os
from bs4 import BeautifulSoup as bs
import time

## Première série sur Allociné : A knight of the Seven Kindoms

### Première page

In [2]:
# Connexion à la page

headers = {'User-Agent': 'Mozilla/5.0'}

proxies = {"http" : "http://45.194.3.108:8080"}

url = 'https://www.allocine.fr/series/ficheserie-28230/critiques/'

page = requests.get(url=url, headers=headers, proxies=proxies)

try:
    page.raise_for_status()
except Exception as exc:
    print(f'Problème dans la requette web : {exc}')

# Création soup
soup = bs(page.text,'html.parser')

ConnectionError: HTTPSConnectionPool(host='www.allocine.fr', port=443): Max retries exceeded with url: /series/ficheserie-28230/critiques/ (Caused by NameResolutionError("HTTPSConnection(host='www.allocine.fr', port=443): Failed to resolve 'www.allocine.fr' ([Errno 11001] getaddrinfo failed)"))

### Récupérations des notes

In [ ]:
noteSoup = soup.select("span.stareval-note")
note = noteSoup[0].getText().strip()
note

'4,5'

In [ ]:
noteSoup[8].getText().strip()

'4,0'

In [ ]:
notesAllocine = []
[notesAllocine.append(notes.getText().strip()) for notes in noteSoup]

print(notesAllocine)

['4,5', '5,0', '5,0', '5,0', '5,0', '4,5', '5,0', '5,0', '4,0', '5,0', '4,0', '4,5', '2,5', '4,5', '5,0']


In [ ]:
# Remplacer les "," par "."

notesAllocine = [notes.replace(",", ".") for notes in notesAllocine]
notesAllocine

['4.5',
 '5.0',
 '5.0',
 '5.0',
 '5.0',
 '4.5',
 '5.0',
 '5.0',
 '4.0',
 '5.0',
 '4.0',
 '4.5',
 '2.5',
 '4.5',
 '5.0']

### Récupération des avis

In [ ]:
avisSoup = soup.select("div.content-txt.review-card-content")
avis = avisSoup[0].getText().strip()
avis

"Seule le format est décevant. Seulement 6 épisodes de 30 minutes qui sortent chaque semaine, c'est difficile psychologiquement. Sinon j'adore ! Beaucoup plus léger que GOT et House of the dragon, ce Knight of the seven kingdoms est probablement une masterclass en devenir ! La suite nous le dira. Vraiment hâte de découvrir tout le potentiel de sir Duncan le grand !"

In [ ]:
avisAllocine = []
[avisAllocine.append(avis.getText(strip=True)) for avis in avisSoup]

print(avisAllocine)

["Seule le format est décevant. Seulement 6 épisodes de 30 minutes qui sortent chaque semaine, c'est difficile psychologiquement.Sinon j'adore ! Beaucoup plus léger que GOT et House of the dragon, ce Knight of the seven kingdoms est probablement une masterclass en devenir ! La suite nous le dira. Vraiment hâte de découvrir tout le potentiel de sir Duncan le grand !", "Très bonne série. Des épisodes trop court \uf60d très fidèle dans l'esprit a la nouvelle. Nous avons regardé les trois premiers épisodes d'une traite. C'est parfait", 'J’y ai trouvé du génie ! Pour ceux qui disent «\xa0c’est moue\xa0», revoyez l’épisode 1 de Game of thrones, à l’époque, beaucoup s’était découragé. Moi j’ai adoré !Update suite au visionnage de l’épisode 4, perso j’ai eu la larme à l’œil, c’est magnifique !', "J'en suis au 4ème épisode et pour l'instant j'adore.Mon seul regret c'est que les épisodes sont trop courts (entre 30 et 35 minutes).Je trouve le scénario et les personnages très intéressants et bien 

In [ ]:
# Nettoyage des caractères spéciaux html du type '\uf449'

avisAllocine = [re.sub(r'[\x00-\x1f\x7f\xa0\uf000-\uffff]', ' ', avis) for avis in avisAllocine]

avisAllocine

["Seule le format est décevant. Seulement 6 épisodes de 30 minutes qui sortent chaque semaine, c'est difficile psychologiquement.Sinon j'adore ! Beaucoup plus léger que GOT et House of the dragon, ce Knight of the seven kingdoms est probablement une masterclass en devenir ! La suite nous le dira. Vraiment hâte de découvrir tout le potentiel de sir Duncan le grand !",
 "Très bonne série. Des épisodes trop court   très fidèle dans l'esprit a la nouvelle. Nous avons regardé les trois premiers épisodes d'une traite. C'est parfait",
 'J’y ai trouvé du génie ! Pour ceux qui disent « c’est moue », revoyez l’épisode 1 de Game of thrones, à l’époque, beaucoup s’était découragé. Moi j’ai adoré !Update suite au visionnage de l’épisode 4, perso j’ai eu la larme à l’œil, c’est magnifique !',
 "J'en suis au 4ème épisode et pour l'instant j'adore.Mon seul regret c'est que les épisodes sont trop courts (entre 30 et 35 minutes).Je trouve le scénario et les personnages très intéressants et bien écrit. A

### Récupérations des dates

In [ ]:
dateSoup = soup.select("span.review-card-meta-date.light")
date = dateSoup[0].getText().strip()
date

'Publiée le 8 février 2026'

In [ ]:
dateAllocine = []
[dateAllocine.append(date.getText(strip=True)) for date in dateSoup]

print(dateAllocine)

['Publiée le 8 février 2026', 'Publiée le 7 février 2026', 'Publiée le 7 février 2026', 'Publiée le 6 février 2026', 'Publiée le 10 février 2026', 'Publiée le 27 janvier 2026', 'Publiée le 19 février 2026', 'Publiée le 3 février 2026', 'Publiée le 8 février 2026', 'Publiée le 20 janvier 2026', 'Publiée le 20 janvier 2026', 'Publiée le 5 mars 2026', 'Publiée le 25 février 2026', 'Publiée le 22 janvier 2026', 'Publiée le 20 janvier 2026']


In [ ]:
# Supprimer 'Publiée le' avant la date

dateAllocine = [date.replace("Publiée le ", "") for date in dateAllocine]
dateAllocine

['8 février 2026',
 '7 février 2026',
 '7 février 2026',
 '6 février 2026',
 '10 février 2026',
 '27 janvier 2026',
 '19 février 2026',
 '3 février 2026',
 '8 février 2026',
 '20 janvier 2026',
 '20 janvier 2026',
 '5 mars 2026',
 '25 février 2026',
 '22 janvier 2026',
 '20 janvier 2026']

## Récupération des données sur les autres pages

In [ ]:
# Création du la soupe sur la barre de navigation
suivantSoup = soup.select("section.section.section-wrap.gd-2-cols.gd-gap-30 > div > nav")
nav = suivantSoup[0]
nav

<nav class="pagination cf"><span class="button button-md button-primary-full button-left button-disabled"><i class="icon icon-left icon-arrow-left"></i><span class="txt">Précédente</span></span><span class="ACrL3NACrlcmllcy9maWNoZXNlcmllLTI4MjMwL2NyaXRpcXVlcy8/cGFnZT0y button button-md button-primary-full button-right"><span class="txt">Suivante</span><i class="icon icon-right icon-arrow-right-a"></i></span><div class="pagination-item-holder"><span class="button button-md item current-item"> 1 </span><a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=2">2</a><a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=3">3</a><a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=4">4</a><a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=5">5</a><a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=6">6</a><a class="button button-md item" href="/series/fichese

In [ ]:
# Les liens sont liées aux balises 'a'
page_link = nav.find_all('a')
page_link

[<a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=2">2</a>,
 <a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=3">3</a>,
 <a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=4">4</a>,
 <a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=5">5</a>,
 <a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=6">6</a>,
 <a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=7">7</a>,
 <a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=8">8</a>,
 <a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=9">9</a>,
 <a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=10">10</a>,
 <a class="button button-md item" href="/series/ficheserie-28230/critiques/?page=14">14</a>]

In [ ]:
# Compter le nombre de pages

nb_pages = len(page_link) + 1
print(nb_pages)


11


In [ ]:
# Vérification des liens URL

relative_url = "https://www.allocine.fr"

for i in range(2, nb_pages+1):  # +1 car Python compte à patir de 0
   print(f"{relative_url}/series/ficheserie-28230/critiques/?page={i}")

https://www.allocine.fr/series/ficheserie-28230/critiques/?page=2
https://www.allocine.fr/series/ficheserie-28230/critiques/?page=3
https://www.allocine.fr/series/ficheserie-28230/critiques/?page=4
https://www.allocine.fr/series/ficheserie-28230/critiques/?page=5
https://www.allocine.fr/series/ficheserie-28230/critiques/?page=6
https://www.allocine.fr/series/ficheserie-28230/critiques/?page=7
https://www.allocine.fr/series/ficheserie-28230/critiques/?page=8
https://www.allocine.fr/series/ficheserie-28230/critiques/?page=9
https://www.allocine.fr/series/ficheserie-28230/critiques/?page=10
https://www.allocine.fr/series/ficheserie-28230/critiques/?page=11


In [ ]:
# Récupération des données sur toutes les pages
for i in range(2, nb_pages+1): 
   full_url = f"{relative_url}/series/ficheserie-28230/critiques/?page={i}"
   page = requests.get(full_url, headers=headers, proxies=proxies)

   try:
    page.raise_for_status()
   except Exception as exc:
    print(f'Problème dans la requette web : {exc}')

   soup = bs(page.text, "html.parser")

   noteSoup = soup.select("span.stareval-note")
   avisSoup = soup.select("div.content-txt.review-card-content")
   dateSoup = soup.select("span.review-card-meta-date.light")

   [notesAllocine.append(notes.getText().strip()) for notes in noteSoup]
   [avisAllocine.append(avis.getText(strip=True)) for avis in avisSoup]
   [dateAllocine.append(date.getText(strip=True)) for date in dateSoup]

   time.sleep(1)

notesAllocine = [notes.replace(",", ".") for notes in notesAllocine]
avisAllocine = [re.sub(r'[\x00-\x1f\x7f\xa0\uf000-\uffff]', ' ', avis) for avis in avisAllocine]
dateAllocine = [date.replace("Publiée le ", "") for date in dateAllocine]

print(notesAllocine)
print(avisAllocine)
print(dateAllocine)


['4.5', '5.0', '5.0', '5.0', '5.0', '4.5', '5.0', '5.0', '4.0', '5.0', '4.0', '4.5', '2.5', '4.5', '5.0', '5.0', '4.0', '2.0', '4.5', '5.0', '5.0', '3.5', '5.0', '5.0', '2.0', '5.0', '5.0', '5.0', '1.0', '2.0', '5.0', '1.0', '4.0', '5.0', '5.0', '4.0', '1.0', '2.5', '2.5', '0.5', '2.5', '4.0', '0.5', '5.0', '3.0', '4.0', '2.0', '1.0', '2.0', '2.0', '5.0', '3.5', '4.5', '5.0', '4.0', '4.0', '5.0', '5.0', '5.0', '5.0', '5.0', '5.0', '5.0', '3.5', '2.0', '5.0', '5.0', '4.5', '5.0', '4.5', '5.0', '0.5', '3.5', '4.0', '3.0', '3.0', '3.5', '0.5', '3.5', '3.0', '3.5', '0.5', '2.0', '4.5', '3.0', '2.5', '5.0', '1.5', '2.0', '1.0', '4.0', '4.0', '3.5', '1.0', '4.0', '3.5', '4.0', '2.0', '1.5', '4.5', '5.0', '5.0', '2.0', '5.0', '5.0', '5.0', '3.0', '5.0', '4.0', '3.5', '5.0', '3.5', '3.0', '5.0', '5.0', '5.0', '3.5', '5.0', '4.0', '4.5', '1.5', '0.5', '3.0', '1.0', '5.0', '4.0', '3.0', '1.5', '4.5', '1.5', '0.5', '0.5', '5.0', '5.0', '5.0', '4.0', '4.0', '5.0', '3.0', '5.0', '5.0', '5.0', '0.5'

In [ ]:
# Création series_id (je reprends le code que l'on retrouve dans l'URL)
find_series_id = re.compile(r'\d+')
allocine_id = find_series_id.findall(url)[0]


## Création DataFrame

In [ ]:
# Vérification nombre de lignes par liste
print(len(notesAllocine))
print(len(avisAllocine))
print(len(dateAllocine))

colums = ['notesAllocine', 'avis', 'dates']

df_knight = pd.DataFrame(list(zip(notesAllocine, avisAllocine, dateAllocine))
                         , columns=colums)
df_knight["series"] = "A Knight of the Seven Kingdoms"
df_knight["allocine_id"] = allocine_id
df_knight.head()

165
165
165


,notesAllocine,avis,dates,series,allocine_id
0,4.5,Seule le format est décevant. Seulement 6 épis...,8 février 2026,A Knight of the Seven Kingdoms,28230
1,5.0,Très bonne série. Des épisodes trop court tr...,7 février 2026,A Knight of the Seven Kingdoms,28230
2,5.0,J’y ai trouvé du génie ! Pour ceux qui disent ...,7 février 2026,A Knight of the Seven Kingdoms,28230
3,5.0,J'en suis au 4ème épisode et pour l'instant j'...,6 février 2026,A Knight of the Seven Kingdoms,28230
4,5.0,"Encore un grand cru, un bijoux, du caviar, ou ...",10 février 2026,A Knight of the Seven Kingdoms,28230


# Série 2 : King & Conqueror

## Première page

In [ ]:
# Connexion à la page
url = 'https://www.allocine.fr/series/ficheserie-35604/critiques/'
page = requests.get(url, headers=headers, proxies=proxies)

try:
    page.raise_for_status()
except Exception as exc:
    print(f'Problème dans la requette web : {exc}')

# Création soups
soup = bs(page.text,'html.parser')
noteSoup = soup.select("span.stareval-note")
avisSoup = soup.select("div.content-txt.review-card-content")
dateSoup = soup.select("span.review-card-meta-date.light")

# Récupération note et premier nettoyage
notesAllocine = []
[notesAllocine.append(notes.getText().strip()) for notes in noteSoup]

# Récupération avis et premier nettoyage
avisAllocine = []
[avisAllocine.append(avis.getText(strip=True)) for avis in avisSoup]

# Récupération date et premier nettoyage
dateAllocine = []
[dateAllocine.append(date.getText(strip=True)) for date in dateSoup]

print(notesAllocine)
print(avisAllocine)
print(dateAllocine)


['0,5', '1,5', '0,5', '1,5', '1,5', '2,0', '1,5', '0,5', '2,0', '1,0', '4,0', '1,5', '3,0', '1,0', '5,0']
["Bon c'est la BBC, donc il fallait s'y attendre... Encore une fois, une série qui aurait pu être bien si elle n'avait pas été gâchée par un agenda politique. À quel moment les scénaristes se sont dit que ce serait une idée brillante si les Normands et anglais du début début du moyen âge étaient représentés par des africains... C'est insultant pour tout le monde et ça détruit toute forme de crédibilité. A fuir, donc.", "J'ai voulu tenter, et j'ai perdu mon temps. La série est cheap plus que de raison, des anachronismes à tout bout de champ pour encore une fois nous faire passer, nous français, pour des guignols qui ne savaient pas se battre (faut-il rappeler que nous sommes la nation à avoir remporté le plus de batailles). Au-delà de l'aspect politique de la série, et si on ne se concentre que sur le visuel, c'est vraiment très très laid, surtout l'étalonnage, tenez-vous prêts à pr

## Autres pages

In [ ]:
# Récupération des données sur toutes les pages

suivantSoup = soup.select_one("section.section.section-wrap.gd-2-cols.gd-gap-30 > div > nav")
page_link = suivantSoup.find_all('a')
nb_pages = len(page_link) + 1

for i in range(2, nb_pages+1):  # +1 car Python compte à patir de 0
   full_url = f"{relative_url}/series/ficheserie-35604/critiques/?page={i}"
   page = requests.get(full_url, headers=headers, proxies=proxies)

   try:
    page.raise_for_status()
   except Exception as exc:
    print(f'Problème dans la requette web : {exc}')

   soup = bs(page.text, "html.parser")

   noteSoup = soup.select("span.stareval-note")
   avisSoup = soup.select("div.content-txt.review-card-content")
   dateSoup = soup.select("span.review-card-meta-date.light")

   [notesAllocine.append(notes.getText().strip()) for notes in noteSoup]
   [avisAllocine.append(avis.getText(strip=True)) for avis in avisSoup]
   [dateAllocine.append(date.getText(strip=True)) for date in dateSoup]

   time.sleep(1)

notesAllocine = [notes.replace(",", ".") for notes in notesAllocine]
avisAllocine = [re.sub(r'[\x00-\x1f\x7f\xa0\uf000-\uffff]', ' ', avis) for avis in avisAllocine]
dateAllocine = [date.replace("Publiée le ", "") for date in dateAllocine]

print(notesAllocine)
print(avisAllocine)
print(dateAllocine)

['0.5', '1.5', '0.5', '1.5', '1.5', '2.0', '1.5', '0.5', '2.0', '1.0', '4.0', '1.5', '3.0', '1.0', '5.0', '1.0', '0.5', '2.5', '4.0', '1.5', '1.0', '2.5', '3.0', '1.0', '5.0', '4.0', '1.0', '4.5', '1.5', '4.0', '0.5', '1.0', '1.5', '1.0']
["Bon c'est la BBC, donc il fallait s'y attendre... Encore une fois, une série qui aurait pu être bien si elle n'avait pas été gâchée par un agenda politique. À quel moment les scénaristes se sont dit que ce serait une idée brillante si les Normands et anglais du début début du moyen âge étaient représentés par des africains... C'est insultant pour tout le monde et ça détruit toute forme de crédibilité. A fuir, donc.", "J'ai voulu tenter, et j'ai perdu mon temps. La série est cheap plus que de raison, des anachronismes à tout bout de champ pour encore une fois nous faire passer, nous français, pour des guignols qui ne savaient pas se battre (faut-il rappeler que nous sommes la nation à avoir remporté le plus de batailles). Au-delà de l'aspect politiqu

In [ ]:
# Création series_id (je reprends le code en 5 chiffres que l'on retrouve dans l'URL)
find_series_id = re.compile(r'\d+')
allocine_id = find_series_id.findall(url)[0]

# Création Dataframe Global

In [ ]:
# Vérification nombre de lignes par liste
print(len(notesAllocine))
print(len(avisAllocine))
print(len(dateAllocine))

columns = ['notesAllocine', 'avis', 'dates']

df_king = pd.DataFrame(list(zip(notesAllocine, avisAllocine, dateAllocine)),
                        columns = columns)
df_king["series"] = "King & Conqueror"
df_king["allocine_id"] = allocine_id
df_king.head()

34
34
34


,notesAllocine,avis,dates,series,allocine_id
0,0.5,"Bon c'est la BBC, donc il fallait s'y attendre...",22 septembre 2025,King & Conqueror,35604
1,1.5,"J'ai voulu tenter, et j'ai perdu mon temps. La...",5 mars 2026,King & Conqueror,35604
2,0.5,Cette production a de nombreux problèmes de ré...,1 septembre 2025,King & Conqueror,35604
3,1.5,Le rappel à Game of Throne ne tient pas longte...,1 septembre 2025,King & Conqueror,35604
4,1.5,"bien réalisé, bien joué, mais l'éternelle mauv...",1 septembre 2025,King & Conqueror,35604


In [ ]:
# Concaténation et conversion des colonnes
df_allocine = pd.concat([df_knight, df_king], ignore_index=True)

# Conversion de la colonne notesAllociné en nombre décimal
df_allocine = df_allocine.astype({'notesAllocine' : 'float',
                                  'allocine_id' : 'int'})

# Pour la date : traduction anglais pour permettre la conversion.
mois = {
    'janvier': 'January', 'février': 'February', 'mars': 'March',
    'avril': 'April', 'mai': 'May', 'juin': 'June',
    'juillet': 'July', 'août': 'August', 'septembre': 'September',
    'octobre': 'October', 'novembre': 'November', 'décembre': 'December'
}

df_allocine['dates'] = df_allocine['dates'].replace(mois, regex=True)

# Conversion au format dates
df_allocine['dates'] = pd.to_datetime(df_allocine['dates'], format='%d %B %Y')


In [ ]:
# Création review_id : le code allociné + la'index
df_allocine["review_id"] = df_allocine["allocine_id"].astype(str) + (df_allocine.index +1 ).astype(str)

# Colonne avec le nom de la plateforme en prévision de la fusion
df_allocine["plateforme"] = "Allocine"

In [ ]:
# Vérifications

display(df_allocine.head())
print() # Saut à la ligne
print(df_allocine.info())

,notesAllocine,avis,dates,series,allocine_id,review_id,plateforme
0,4.5,Seule le format est décevant. Seulement 6 épis...,2026-02-08,A Knight of the Seven Kingdoms,28230,282301,Allocine
1,5.0,Très bonne série. Des épisodes trop court tr...,2026-02-07,A Knight of the Seven Kingdoms,28230,282302,Allocine
2,5.0,J’y ai trouvé du génie ! Pour ceux qui disent ...,2026-02-07,A Knight of the Seven Kingdoms,28230,282303,Allocine
3,5.0,J'en suis au 4ème épisode et pour l'instant j'...,2026-02-06,A Knight of the Seven Kingdoms,28230,282304,Allocine
4,5.0,"Encore un grand cru, un bijoux, du caviar, ou ...",2026-02-10,A Knight of the Seven Kingdoms,28230,282305,Allocine



<class 'pandas.DataFrame'>
RangeIndex: 199 entries, 0 to 198
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   notesAllocine  199 non-null    float64       
 1   avis           199 non-null    str           
 2   dates          199 non-null    datetime64[us]
 3   series         199 non-null    str           
 4   allocine_id    199 non-null    int64         
 5   review_id      199 non-null    str           
 6   plateforme     199 non-null    str           
dtypes: datetime64[us](1), float64(1), int64(1), str(4)
memory usage: 11.0 KB
None


In [ ]:
# Export en csv

# Un point = dossier actuel, deux points = dossier parent
output_path = '../data/data_allocine.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_allocine.to_csv(output_path, index=False)

# Scrapping informations séries

Nous allons récupérer les informations sur la série pour créer une nouvelle table 'série'.

In [ ]:
# Code de test pour trouver les balises
'''url_def = 'https://www.allocine.fr/series/ficheserie_gen_cserie=28230.html'
 
page_def = requests.get(url_def)

try:
  page_def.raise_for_status()
except Exception as exc:
  print(f'Problème dans la requette web : {exc}')

# Création soup
soup_def = bs(page_def.text,'html.parser')

# Récupération du code allociné
find_series_id = re.compile(r'\d+')
allocine_id = find_series_id.findall(url_def)[0]

# Récupération du titre
titleSoup = soup_def.select("div.titlebar-title.titlebar-title-xl")
title = titleSoup[0].getText()

# Récupération des genres
genresSoup = soup_def.select("div.meta-body-info span.dark-grey-link")
genresList = [genre.getText(strip=True) for genre in genresSoup]
genres = "|".join(genresList)

# Récupération des réalisateurs
directorsSoup = soup_def.select("div.meta-body-direction a.dark-grey-link")
directorsList = [director.getText(strip=True) for director in directorsSoup]
directors = "|".join(directorsList)

# Récupération des acteurs
actorsSoup = soup_def.select("div.meta-body-actor span.dark-grey-link")
actorsList = [actor.getText(strip=True) for actor in actorsSoup]
actors = "|".join(actorsList)

# Récupération de la nationalité
nationalitySoup = soup_def.select("div.meta-body-nationality span.dark-grey-link")
nationalityList = [nationality.getText(strip=True) for nationality in nationalitySoup]
nationalities = "|".join(nationalityList)

data = {'allocine_id' : allocine_id,
        'title' : title,
        'genres' : genres,
        'directors' : directors,
        'actors' : actors,
        'nationalities' : nationalities}

df_series = pd.DataFrame([data])

df_series.to_csv('df_info_series.csv', index=False)
'''

<>:15: SyntaxWarning: invalid escape sequence '\d'
<>:15: SyntaxWarning: invalid escape sequence '\d'
C:\Users\CLFO0303\AppData\Local\Temp\ipykernel_14356\3470842759.py:15: SyntaxWarning: invalid escape sequence '\d'
  find_series_id = re.compile(r'\d+')


'url_def = \'https://www.allocine.fr/series/ficheserie_gen_cserie=28230.html\'\n\npage_def = requests.get(url_def)\n\ntry:\n  page_def.raise_for_status()\nexcept Exception as exc:\n  print(f\'Problème dans la requette web : {exc}\')\n\n# Création soup\nsoup_def = bs(page_def.text,\'html.parser\')\n\n# Récupération du code allociné\nfind_series_id = re.compile(r\'\\d+\')\nallocine_id = find_series_id.findall(url_def)[0]\n\n# Récupération du titre\ntitleSoup = soup_def.select("div.titlebar-title.titlebar-title-xl")\ntitle = titleSoup[0].getText()\n\n# Récupération des genres\ngenresSoup = soup_def.select("div.meta-body-info span.dark-grey-link")\ngenresList = [genre.getText(strip=True) for genre in genresSoup]\ngenres = "|".join(genresList)\n\n# Récupération des réalisateurs\ndirectorsSoup = soup_def.select("div.meta-body-direction a.dark-grey-link")\ndirectorsList = [director.getText(strip=True) for director in directorsSoup]\ndirectors = "|".join(directorsList)\n\n# Récupération des ac

In [ ]:
# Code propre avec les définitions.

def info_series(allocine_code : int) -> dict:
    '''Renvoie les informations de la série sur Allocine sous forme de dict.
       Utiliser le code allociné dans l'URL de la série.
       Return None
    '''

    url = f'https://www.allocine.fr/series/ficheserie_gen_cserie={allocine_code}.html'
    page = requests.get(url, headers=headers, proxies=proxies)

    try:
      page.raise_for_status()
    except Exception as exc:
      print(f'Problème pour le code {allocine_code} : {exc}')
      return None

    # Création soup
    soup = bs(page.text,'html.parser')

    # Récupération titre
    title_tag = soup.select_one("div.titlebar-title.titlebar-title-xl")

    # Récupération affiche
    meta_image = soup.select_one('meta[property="og:image"]')
    image_url = meta_image["content"] if meta_image else None

    def extract_text(selector : str):
       '''Extrait et joint les textes d'un sélecteur CSS'''
       return "|".join(el.getText(strip=True) for el in soup.select(selector))
   
    return {
       'allocine_id' : allocine_code,
       'title' : title_tag.getText(strip=True) if title_tag else None,
       'genres' : extract_text("div.meta-body-info span.dark-grey-link"),
       'directors' : extract_text("div.meta-body-direction a.dark-grey-link"),
       'actors' : extract_text("div.meta-body-actor span.dark-grey-link"),
       'nationalities' : extract_text("div.meta-body-nationality span.dark-grey-link"),
       'affiche' : image_url
    }

# Utilisation du walrus operator pour s'assurer que l'itération continue si le code n'est pas trouvé
rows = [
   result for code in df_allocine['allocine_id'].unique()
   if (result := info_series(code)) is not None
]
# Le Walrus opérator fait le test ET stocke la valeur dans result

df_series = pd.DataFrame(rows)

# Export csv
# Un point = dossier actuel, deux points = dossier parent
output_path = '../data//info_series.csv'
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df_series.to_csv(output_path, index=False)
    